# Polyp Detection — Higher Resolution Retraining

First hypothesis test following error analysis (notebook 04b):
retraining at imgsz=960 to give small polyps more pixels to work with.

Error analysis showed missed polyps are 59% smaller (median area)
than detected ones. The hypothesis: at 640px, small polyps occupy
too few pixels for the model to learn reliable features. At 960px,
those same polyps would occupy ~2.25x more pixels.

---

## What this notebook does

1. Retrains YOLOv8m-seg at imgsz=960 with batch=4 (memory constraint)
2. Evaluates on Kvasir-SEG test and CVC-ClinicDB at conf=0.30
3. Re-runs the missed-polyp size analysis from notebook 04b
4. Compares baseline (640) vs higher-res (960) directly

## Result

The hypothesis did not hold. Miss rate increased from 11.3% to 14.1%.
Reducing batch from 8 to 4 to fit the larger resolution hurt training
stability more than the resolution gain helped.

This negative result directly informed the next experiment (notebook 04d).

## Output

- models/imgsz960/weights/best.pt
- results/metrics/imgsz960_metrics.json
- results/figures/imgsz960_size_comparison.png

In [ ]:
# Import libraries

import json
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\polyp-detection-safety-first\repo"

PATHS = {
    "dataset_yaml":  os.path.join(BASE_DIR, "configs", "dataset.yaml"),
    "cvc_eval_yaml": os.path.join(BASE_DIR, "configs", "dataset_cvc_eval.yaml"),
    "cvc_images":    os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "images"),
    "cvc_labels":    os.path.join(BASE_DIR, "data", "yolo_format", "cvc_test", "labels"),
    "metrics":       os.path.join(BASE_DIR, "results", "metrics"),
    "figures":       os.path.join(BASE_DIR, "results", "figures"),
}

os.makedirs(PATHS["metrics"], exist_ok=True)
os.makedirs(PATHS["figures"], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Training Hyperparameters
# Higher-resolution training configuration
#
# imgsz=960 vs. 640 baseline: small polyps occupy more pixels at
# this resolution, which should help the model learn their features.
# batch is reduced from 8 to 4 to keep memory usage within the
# 8GB VRAM budget - higher resolution images use significantly
# more memory per sample.

TRAIN_CONFIG = {
    "model":     "yolov8m-seg.pt",
    "data":      PATHS["dataset_yaml"],
    "epochs":    100,
    "imgsz":     960,
    "batch":     4,
    "patience":  15,
    "optimizer": "AdamW",
    "lr0":       0.001,
    "seed":      42,
    "amp":       True,
    "project":   os.path.join(BASE_DIR, "models"),
    "name":      "imgsz960",
    "exist_ok":  True,
    "verbose":   True,
}

print("Training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key:10s}: {value}")

In [ ]:
# Run Training
# Expect 60-90 minutes: smaller batch + larger images mean more
# compute per step than the 27-minute baseline run.
# If OOM error occurs, reduce batch to 2 in cell3 and re-run.

model = YOLO(TRAIN_CONFIG["model"])

start_time = time.time()

train_results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    patience=TRAIN_CONFIG["patience"],
    optimizer=TRAIN_CONFIG["optimizer"],
    lr0=TRAIN_CONFIG["lr0"],
    seed=TRAIN_CONFIG["seed"],
    amp=TRAIN_CONFIG["amp"],
    project=TRAIN_CONFIG["project"],
    name=TRAIN_CONFIG["name"],
    exist_ok=TRAIN_CONFIG["exist_ok"],
    verbose=TRAIN_CONFIG["verbose"],
)

elapsed_min = (time.time() - start_time) / 60
print(f"\nTraining finished in {elapsed_min:.1f} minutes")

In [ ]:
# Load Model and Evaluate (Same Framework as Notebook 04)
# Load the higher-resolution model and evaluate using the same
# metrics framework as notebook 04, for direct comparison

weights_path = os.path.join(
    TRAIN_CONFIG["project"], TRAIN_CONFIG["name"], "weights", "best.pt"
)
model_960 = YOLO(weights_path)
print(f"Loaded: {weights_path}")

kvasir_results = model_960.val(
    data=PATHS["dataset_yaml"],
    split="test",
    imgsz=TRAIN_CONFIG["imgsz"],
    conf=0.30,  # same operating threshold selected in notebook 04
)

cvc_results = model_960.val(
    data=PATHS["cvc_eval_yaml"],
    split="val",
    imgsz=TRAIN_CONFIG["imgsz"],
    conf=0.30,
)

print("\nKVASIR-SEG TEST (imgsz=960, conf=0.30)")
print("-" * 40)
print(f"Recall:    {kvasir_results.box.mr:.4f}")
print(f"Precision: {kvasir_results.box.mp:.4f}")
print(f"mAP50:     {kvasir_results.box.map50:.4f}")

print("\nCVC-CLINICDB CROSS-DATASET (imgsz=960, conf=0.30)")
print("-" * 40)
print(f"Recall:    {cvc_results.box.mr:.4f}")
print(f"Precision: {cvc_results.box.mp:.4f}")
print(f"mAP50:     {cvc_results.box.map50:.4f}")

In [ ]:
# Re-run Error Analysis (Same Framework as Notebook 04b)
# Same missed-detection analysis as notebook 04b, but with the
# imgsz=960 model, to verify whether the small-polyp problem
# actually improved

def read_yolo_label_as_boxes(label_path, img_width, img_height):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            coords = list(map(float, parts[1:]))
            points = np.array(coords).reshape(-1, 2)
            points[:, 0] *= img_width
            points[:, 1] *= img_height
            x_min, y_min = points.min(axis=0)
            x_max, y_max = points.max(axis=0)
            boxes.append([x_min, y_min, x_max, y_max])
    return boxes


def box_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_area = max(0, x2 - x1) * max(0, y2 - y1)
    if inter_area == 0:
        return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter_area / (area1 + area2 - inter_area)


IOU_MATCH_THRESHOLD = 0.3
image_files = sorted(os.listdir(PATHS["cvc_images"]))

missed_960   = []
detected_960 = []

for fname in image_files:
    stem = os.path.splitext(fname)[0]
    img_path   = os.path.join(PATHS["cvc_images"], fname)
    label_path = os.path.join(PATHS["cvc_labels"], stem + ".txt")

    img = Image.open(img_path)
    w, h = img.size

    gt_boxes = read_yolo_label_as_boxes(label_path, w, h)
    if not gt_boxes:
        continue

    pred = model_960.predict(img_path, conf=0.10, imgsz=960, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy().tolist() if len(pred.boxes) > 0 else []

    for gt_box in gt_boxes:
        gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
        gt_area_ratio = gt_area / (w * h)
        best_iou = max((box_iou(gt_box, pb) for pb in pred_boxes), default=0.0)

        record = {"image": fname, "area_ratio": gt_area_ratio, "best_iou": best_iou}
        if best_iou < IOU_MATCH_THRESHOLD:
            missed_960.append(record)
        else:
            detected_960.append(record)

miss_rate_960 = len(missed_960) / (len(missed_960) + len(detected_960))

print(f"Total ground-truth polyps: {len(missed_960) + len(detected_960)}")
print(f"Missed (imgsz=960): {len(missed_960)}")
print(f"Miss rate (imgsz=960): {miss_rate_960*100:.1f}%")
print(f"Miss rate (imgsz=640, from notebook 04b): 11.3%")

In [ ]:
# Compare Size Distributions: 640 vs 960
# Did the size gap between missed and detected polyps shrink
# at the higher resolution? This is the direct test of whether
# imgsz=960 actually fixed the small-object problem.

missed_areas_960   = [r["area_ratio"] for r in missed_960]
detected_areas_960 = [r["area_ratio"] for r in detected_960]

print("POLYP AREA COMPARISON (imgsz=960)")
print("-" * 40)
print(f"Detected polyps - median area: {np.median(detected_areas_960)*100:.2f}%")
print(f"Missed polyps   - median area: {np.median(missed_areas_960)*100:.2f}%")

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(
    [detected_areas_960, missed_areas_960],
    tick_labels=["Detected", "Missed"],
)
ax.set_ylabel("Polyp area (fraction of image)")
ax.set_title("Polyp Size: Detected vs Missed (imgsz=960, CVC-ClinicDB)")
ax.grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "imgsz960_size_comparison.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Save Comparison Metrics

comparison = {
    "imgsz_640_baseline": {
        "miss_rate": 0.1130,
        "median_area_missed": 0.0374,
    },
    "imgsz_960": {
        "miss_rate": miss_rate_960,
        "median_area_missed": float(np.median(missed_areas_960)) if missed_areas_960 else None,
        "kvasir_recall": float(kvasir_results.box.mr),
        "kvasir_precision": float(kvasir_results.box.mp),
        "cvc_recall": float(cvc_results.box.mr),
        "cvc_precision": float(cvc_results.box.mp),
    },
}

metrics_path = os.path.join(PATHS["metrics"], "imgsz960_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(comparison, f, indent=2)

print(f"Saved -> {metrics_path}")
print()
print(json.dumps(comparison, indent=2))

In [ ]:
# Summary

print("NOTEBOOK 04c COMPLETE")
print("=" * 50)
print(f"imgsz=640 (baseline) miss rate: 11.3%")
print(f"imgsz=960 miss rate:            {miss_rate_960*100:.1f}%")
print()
improvement = 11.3 - miss_rate_960 * 100
if improvement > 0:
    print(f"Improvement: {improvement:.1f} percentage points")
else:
    print(f"No improvement (or regression): {improvement:.1f} percentage points")
print()
print("CVC-ClinicDB cross-dataset recall:")
print(f"  imgsz=640: 0.752 (from notebook 04)")
print(f"  imgsz=960: {cvc_results.box.mr:.3f}")
print()
print("Next -> 05_evaluation.ipynb")
print("  - Final three-way comparison: baseline / safety-first / imgsz960")
print("  - Decide which configuration becomes the final model")